# 02 – Inference & Submission
Load pre-extracted features + fitted fusion model, generate a submission CSV.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import json
from pathlib import Path

from src.utils.config import load_config
from src.models.fusion import LGBMFusion

ROOT = Path('..')
cfg = load_config(ROOT / 'config.yaml')

FOLD = 0          # change to match which fold checkpoint to use
FEAT_DIR = ROOT / cfg.paths.features_dir / 'test'
MODEL_PATH = ROOT / cfg.paths.checkpoints_dir / f'lgbm_fusion_fold{FOLD}.pkl'
THRESH_PATH = ROOT / cfg.paths.checkpoints_dir / f'lgbm_fusion_fold{FOLD}.threshold.json'
OUT_PATH = ROOT / cfg.paths.submissions_dir / f'submission_fold{FOLD}.csv'

print(f'Feature dir  : {FEAT_DIR}')
print(f'Model path   : {MODEL_PATH}')
print(f'Output       : {OUT_PATH}')

In [ ]:
# Load test features
parts = []
for fname in ['global_feats.npy', 'dinov2_feats.npy', 'physical_feats.npy', 'metadata.npy']:
    fpath = FEAT_DIR / fname
    if fpath.exists():
        arr = np.load(fpath).astype(np.float32)
        parts.append(arr)
        print(f'  {fname}: {arr.shape}')
    else:
        print(f'  MISSING: {fname}')

X_test = np.concatenate(parts, axis=1)
ids = np.load(FEAT_DIR / 'ids.npy', allow_pickle=True)
print(f'X_test: {X_test.shape}  | n_ids: {len(ids)}')

In [ ]:
# Load model and threshold
fusion = LGBMFusion.load(MODEL_PATH)
scores = fusion.predict(X_test)

if THRESH_PATH.exists():
    with open(THRESH_PATH) as f:
        meta = json.load(f)
    print(f'Val FREUID={meta["freuid"]:.4f}  threshold={meta["threshold"]:.4f}')

print(f'Score stats → min={scores.min():.4f}  max={scores.max():.4f}  mean={scores.mean():.4f}')

In [ ]:
import matplotlib.pyplot as plt
plt.hist(scores, bins=20, color='steelblue', edgecolor='k')
plt.xlabel('Fraud score'); plt.ylabel('Count')
plt.title('Distribution of predicted fraud scores')
plt.show()

In [ ]:
# Build submission DataFrame
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
sub = pd.DataFrame({'id': ids, 'label': scores})
sub.to_csv(OUT_PATH, index=False)
print(f'Saved {len(sub)} rows → {OUT_PATH}')
sub.head(10)

In [ ]:
# --- Ensemble: average scores from multiple fold models ---
# Uncomment and adjust paths when you have multi-fold models

# fold_scores = []
# for fold in range(cfg.data.n_folds):
#     mp = ROOT / cfg.paths.checkpoints_dir / f'lgbm_fusion_fold{fold}.pkl'
#     if mp.exists():
#         m = LGBMFusion.load(mp)
#         fold_scores.append(m.predict(X_test))
# ensemble_scores = np.mean(fold_scores, axis=0)
# sub_ens = pd.DataFrame({'id': ids, 'label': ensemble_scores})
# sub_ens.to_csv(ROOT / cfg.paths.submissions_dir / 'submission_ensemble.csv', index=False)
# print(f'Ensemble submission saved.')